In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
import os
os.makedirs("data", exist_ok=True)

In [3]:
import shutil

shutil.move("research2.pdf", "data/research2.pdf")
shutil.move("bookchapter.pdf", "data/bookchapter.pdf")
shutil.move("Python.txt", "data/Python.txt")

'data/Python.txt'

In [4]:
from langchain_core.documents import Document

In [5]:
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source": "https://www.google.com"}
)

In [6]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [7]:
type(sample_doc)

langchain_core.documents.base.Document

In [ ]:
# # Text Data
# from langchain_community.document_loaders.text import TextLoader
# loader = TextLoader("data/Python.txt", encoding="latin-1")

In [ ]:
# document = loader.load()

In [ ]:
# document

In [ ]:
# # PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader
# pdf_loader = PyPDFLoader("data/research2.pdf")
# document = pdf_loader.load()
# document

Ingestion Pipeline

In [8]:
# Data->Document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [9]:
def load_all_pdfs():
  folder_path = "data"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      pdf_path = os.path.join(folder_path, filename)

      try:
        loader = PyPDFLoader(pdf_path)
        doc = loader.load()

        all_docs.extend(doc)
        num_docs += 1
      except Exception as e:
        print(f"Error loading PDF {filename}: {e}")

  print("total_pdfs:", num_docs)
  print("total_pages:", len(all_docs))
  return all_docs

In [10]:
all_pdf_documents = load_all_pdfs()

total_pdfs: 2
total_pages: 48


In [11]:
# Chunks
!pip install langchain_text_splitters


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_doc(documents, chunk_size=500, chunk_overlap=50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [13]:
chunks = split_doc(all_pdf_documents)

In [14]:
len(chunks)

390

Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
class EmbeddingManager:
  def __init__(self, model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("Loading model", self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimension=", self.model.get_sentence_embedding_dimension())
  def generate_embeddings(self, text):
    embeddings = self.model.encode(text)
    print("embedding shape:", embeddings.shape)
    return embeddings

In [18]:
embedding_manager = EmbeddingManager()

Loading model all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimension= 384


/tmp/ipykernel_11461/4022114334.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimension=", self.model.get_sentence_embedding_dimension())


Vector Store

In [19]:
import chromadb
import uuid

In [20]:
class VectorStoreManager:
  def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None

    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory, exist_ok=True)
    #create a client
    self.client = chromadb.PersistentClient(path=self.persist_directory)
    #create the collection
    self.collection = self.client.get_or_create_collection(
      name=self.collection_name,
      metadata={"hnsw:space": "cosine"}  # ← add this
    )

    print("initialize the vector store with collection", self.collection_name)
    print("docs in collection", self.collection.count())

  def add_documents(self, documents, embeddings):
    if(len(documents) != len(embeddings)):
      raise ValueError("number of documents does not match number of embeddings")

    #store -> ids, embedding, document, metadata
    ids = []
    all_metadata = []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate (zip(documents, embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata["doc_index"] = i
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

    self.collection.add(
        ids = ids,
        embeddings = embeddings_list,
        metadatas = all_metadata, # Changed from 'metadata' to 'metadatas'
        documents = documents_content
    )

    print("total documents added in vector store = ", len(documents_content))
    print("docs collection:", self.collection.count())

In [21]:
vector_store = VectorStoreManager()

initialize the vector store with collection pdf_documents
docs in collection 390


In [22]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

embedding shape: (390, 384)
total documents added in vector store =  390
docs collection: 780


Retrievel Pipeline

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class RAGRetriever:
  def __init__(self, embedding_manager, vector_store):
    self.embedding_manager = embedding_manager
    self.vector_store = vector_store

  def retrieve(self, query, top_k=5, score_threshold = 0.0):
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    results = self.vector_store.collection.query(
        query_embeddings = [query_embedding],
        n_results = top_k
    )

    retrieved_docs = []
    if results["documents"] and results["documents"][0]:
      ids = results["ids"][0]
      documents = results["documents"][0]
      metadatas = results["metadatas"][0]
      distances = results["distances"][0]

      for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
        similarity_score = 1-distance # Note: This is a simplified similarity score based on L2 distance

        # Filter based on score_threshold
        # If score_threshold is 0.0, return all top_k results. Otherwise, apply the threshold.
        if score_threshold == 0.0 or similarity_score >= score_threshold:
          retrieved_docs.append(
              {
                  "id": doc_id,
                  "metadata": metadata,
                  "document": document,
                  "distance": distance,
                  "similarity_score": similarity_score,
                  "rank": i+1
              }

          )
    else:
      print("No documents found from vector store query.")

    print(f"Total documents retrieved after filtering: {len(retrieved_docs)}")
    return retrieved_docs

In [25]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)


In [26]:
rag_retriever.retrieve("What is RAG?")

embedding shape: (1, 384)
Total documents retrieved after filtering: 5


[{'id': 'doc_e3354abe-3708-4453-9047-2c21705a68df',
  'metadata': {'total_pages': 21,
   'page': 0,
   'doc_index': 157,
   'content_length': 288,
   'creator': 'LaTeX with hyperref',
   'trapped': '/False',
   'producer': 'pdfTeX-1.40.25',
   'moddate': '2024-03-28T00:54:45+00:00',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'author': '',
   'source': 'data/research2.pdf',
   'keywords': '',
   'page_label': '1',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'title': '',
   'subject': ''},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.2314574122428894,
  'similarity_score': 0.7685425877571106,
  'rank': 1},
 {'id': 'doc_c7a13430-9f13-

Groq

In [27]:
from google.colab import userdata
API_Key_GROQ = userdata.get('GROQ_API_KEY')

In [28]:
!pip install langchain-groq

In [29]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model = "qwen/qwen3-32b",
    temperature=0.1,
    max_tokens = 1024
)

In [30]:
def generate_output(query, retriever, llm, top_k=3):
  results = retriever.retrieve(query, top_k)
  context = "\n".join([doc["document"] for doc in results]) if results else ""

  if not context:
    print("we found no relevant context for the given query")
  # context + query
  prompt = f"""use given context to generate answer for the query
  Context:{context}
  Query: {query} """

  response = llm.invoke([prompt])
  return response.content

In [31]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

embedding shape: (1, 384)
Total documents retrieved after filtering: 3


In [32]:
print(answer)

<think>
Okay, the user is asking, "What is encoder-decoder?" Let me start by recalling what I know about encoder-decoder architectures. They are commonly used in machine learning, especially in tasks like machine translation or text summarization. The encoder processes the input data, converting it into a context vector, and the decoder generates the output based on that vector.

Looking at the provided context, there's mention of PRCA and RECOMP using information extractors and condensers trained with contrastive learning. The encoder is part of their setup, trained with contrastive loss using positive and negative samples. Also, BLIP-2 uses a frozen image encoder with LLMs for visual language tasks. The GSS method stitches audio clips, which might involve encoders for processing audio data.

I need to connect these examples to explain the encoder-decoder structure. The encoder in these contexts is responsible for compressing or extracting essential information from the input (like te

In [36]:
!ls /content/*.ipynb

ls: cannot access '/content/*.ipynb': No such file or directory


In [ ]:
# Step 1: Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Find your notebook
!find /content/drive -name "*.ipynb" 2>/dev/null

In [35]:
import nbformat
import json

# --- Start of Fix ---
# The FileNotFoundError indicates that the system could not find the notebook at the specified path.
# This is likely because the filename or the path is incorrect.
# Please update the 'notebook_path' variable below to the *exact* name and full path
# of the notebook file you are currently working on or wish to download.
# For example, if your notebook is named 'MyProject.ipynb' and is in the root Colab directory,
# you should set it to "/content/MyProject.ipynb".
notebook_path = "/content/RAG_Pipeline_AC.ipynb"  # <--- YOU NEED TO CHANGE THIS TO YOUR NOTEBOOK'S ACTUAL PATH/FILENAME
# --- End of Fix ---

with open(notebook_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

# Fix broken widget metadata
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

# Fix widget state in each cell
for cell in nb.cells:
    if "metadata" in cell and "widgets" in cell.metadata:
        del cell.metadata["widgets"]

with open(notebook_path, "w") as f:
    nbformat.write(nb, f)

# Download the fixed notebook
from google.colab import files
files.download(notebook_path)

print("Done! Check your downloads folder.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/RAG_Pipeline_AC.ipynb'